###Section E

In [0]:
dbutils.widgets.text("catalog", "vp_ms20010501")
catalog = dbutils.widgets.get("catalog")

###E1

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_date AS
SELECT
    CAST(DATE_FORMAT(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
    calendar_date,
    DATE_FORMAT(calendar_date, 'EEEE') AS day_name,
    WEEKOFYEAR(calendar_date) AS iso_week,
    CASE WHEN DAYOFWEEK(calendar_date) IN (1, 7) THEN TRUE ELSE FALSE END AS is_weekend
FROM (
    SELECT EXPLODE(SEQUENCE(DATE '2026-04-01', DATE '2026-04-30', INTERVAL 1 DAY)) AS calendar_date
);
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_station AS
SELECT * FROM {catalog}.silver.stations;
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_charger AS
SELECT
    c.charger_id,
    c.station_id,
    c.connector_type,
    c.max_kw,
    s.city
FROM {catalog}.silver.chargers c
JOIN {catalog}.silver.stations s ON c.station_id = s.station_id;
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.dim_member AS
SELECT
    ROW_NUMBER() OVER (ORDER BY member_id, effective_from) AS member_sk,
    *
FROM {catalog}.silver.members;
""")

###E2

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.fact_sessions AS
WITH base_sessions AS (
    SELECT
        s.session_id,
        CAST(DATE_FORMAT(s.session_date, 'yyyyMMdd') AS INT) AS date_key,
        s.charger_id,
        ch.station_id,
        s.member_id,
        s.status,
        s.time_band,
        s.energy_kwh,
        s.duration_minutes,
        s.idle_minutes,
        s.amount_paid,
        s.session_date,
        ch.connector_type
    FROM {catalog}.silver.sessions s
    JOIN {catalog}.gold.dim_charger ch ON s.charger_id = ch.charger_id
),
matched_members AS (
    SELECT 
        bs.*,
        m.member_sk,
        COALESCE(m.plan, 'GUEST') AS plan_at_session
    FROM base_sessions bs
    LEFT JOIN {catalog}.gold.dim_member m 
        ON bs.member_id = m.member_id
        AND bs.session_date >= m.effective_from 
        AND (bs.session_date < m.effective_to OR m.effective_to IS NULL)
),
fault_counts AS (
    SELECT session_id, COUNT(*) AS fault_count
    FROM {catalog}.silver.session_faults
    GROUP BY session_id
),
tariffs AS (
    SELECT connector_type, time_band, rate_per_kwh, free_idle_minutes, idle_fee_per_min
    FROM {catalog}.silver.tariffs
),
calculated AS (
    SELECT 
        mm.*,
        COALESCE(t.rate_per_kwh, 0.0) AS rate_per_kwh,
        COALESCE(t.free_idle_minutes, 0) AS free_idle_minutes,
        COALESCE(t.idle_fee_per_min, 0.0) AS idle_fee_per_min,
        COALESCE(fc.fault_count, 0) AS fault_count
    FROM matched_members mm
    LEFT JOIN tariffs t ON mm.connector_type = t.connector_type AND mm.time_band = t.time_band
    LEFT JOIN fault_counts fc ON mm.session_id = fc.session_id
)
SELECT 
    session_id,
    date_key,
    charger_id,
    station_id,
    member_id,
    member_sk,
    status,
    time_band,
    energy_kwh,
    duration_minutes,
    idle_minutes,
    amount_paid,
    plan_at_session,
    rate_per_kwh,
    
    -- Billing calculations (applied to COMPLETED sessions only; 0.00 / NULL for others)
    CASE WHEN status = 'COMPLETED' THEN ROUND(energy_kwh * rate_per_kwh, 2) ELSE CAST(0.00 AS DECIMAL(12,2)) END AS energy_charge,
    
    CASE WHEN status = 'COMPLETED' THEN 
        ROUND(ROUND(energy_kwh * rate_per_kwh, 2) * 
            CASE plan_at_session 
                WHEN 'PRO' THEN 0.10 
                WHEN 'FLEET' THEN 0.15 
                ELSE 0.00 
            END, 2)
        ELSE CAST(0.00 AS DECIMAL(12,2)) 
    END AS plan_discount,
    
    CASE WHEN status = 'COMPLETED' THEN 
        ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2)
        ELSE CAST(0.00 AS DECIMAL(12,2)) 
    END AS idle_fee,
    
    CASE WHEN status = 'COMPLETED' THEN 
        ROUND(
            ROUND(energy_kwh * rate_per_kwh, 2) - 
            ROUND(ROUND(energy_kwh * rate_per_kwh, 2) * CASE plan_at_session WHEN 'PRO' THEN 0.10 WHEN 'FLEET' THEN 0.15 ELSE 0.00 END, 2) + 
            ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2), 2
        )
        ELSE CAST(0.00 AS DECIMAL(12,2)) 
    END AS billed_amount,
    
    CASE WHEN status = 'COMPLETED' THEN 
        CASE WHEN ABS(amount_paid - (
            ROUND(energy_kwh * rate_per_kwh, 2) - 
            ROUND(ROUND(energy_kwh * rate_per_kwh, 2) * CASE plan_at_session WHEN 'PRO' THEN 0.10 WHEN 'FLEET' THEN 0.15 ELSE 0.00 END, 2) + 
            ROUND(GREATEST(idle_minutes - free_idle_minutes, 0) * idle_fee_per_min, 2)
        )) <= 0.50 THEN 'OK' ELSE 'MISMATCH' END
        ELSE NULL 
    END AS billing_status,
    
    fault_count
FROM calculated;
""")

###E3

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.gold.agg_daily_station AS
WITH date_station_grid AS (
    SELECT d.calendar_date AS session_date, st.station_id, st.city,
           (SELECT COUNT(*) FROM {catalog}.gold.dim_charger ch WHERE ch.station_id = st.station_id) AS charger_count
    FROM {catalog}.gold.dim_date d
    CROSS JOIN {catalog}.gold.dim_station st
),
session_metrics AS (
    SELECT 
        d.calendar_date AS session_date,
        f.station_id,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN 1 ELSE 0 END) AS completed_sessions,
        SUM(CASE WHEN f.status = 'FAILED' THEN 1 ELSE 0 END) AS failed_sessions,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.energy_kwh ELSE 0 END) AS energy_kwh,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.billed_amount ELSE 0 END) AS billed_amount,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.amount_paid ELSE 0 END) AS amount_paid,
        SUM(CASE WHEN f.status = 'COMPLETED' AND f.billing_status = 'MISMATCH' THEN 1 ELSE 0 END) AS mismatch_sessions,
        SUM(f.fault_count) AS fault_count,
        SUM(CASE WHEN f.status = 'COMPLETED' THEN f.duration_minutes ELSE 0 END) AS completed_duration_minutes
    FROM {catalog}.gold.fact_sessions f
    JOIN {catalog}.gold.dim_date d ON f.date_key = d.date_key
    GROUP BY d.calendar_date, f.station_id
)
SELECT 
    g.session_date,
    g.station_id,
    g.city,
    COALESCE(m.completed_sessions, 0) AS completed_sessions,
    COALESCE(m.failed_sessions, 0) AS failed_sessions,
    COALESCE(m.energy_kwh, CAST(0.000 AS DECIMAL(12,3))) AS energy_kwh,
    COALESCE(m.billed_amount, CAST(0.00 AS DECIMAL(12,2))) AS billed_amount,
    COALESCE(m.amount_paid, CAST(0.00 AS DECIMAL(12,2))) AS amount_paid,
    COALESCE(m.mismatch_sessions, 0) AS mismatch_sessions,
    COALESCE(m.fault_count, 0) AS fault_count,
    CAST(
        CASE WHEN g.charger_count > 0 
        THEN (100.0 * COALESCE(m.completed_duration_minutes, 0)) / (g.charger_count * 1440.0) 
        ELSE 0.0 END AS DECIMAL(5,2)
    ) AS utilisation_pct
FROM date_station_grid g
LEFT JOIN session_metrics m ON g.session_date = m.session_date AND g.station_id = m.station_id;
""")

###E4

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.v_kpi_summary AS
SELECT 
    SUM(completed_sessions) AS completed_sessions,
    SUM(failed_sessions) AS failed_sessions,
    CAST((SUM(completed_sessions) * 100.0 / NULLIF(SUM(completed_sessions) + SUM(failed_sessions), 0)) AS DECIMAL(5,2)) AS success_rate_pct,
    SUM(energy_kwh) AS total_energy_kwh,
    SUM(billed_amount) AS total_billed,
    SUM(amount_paid) AS total_paid,
    SUM(billed_amount - amount_paid) AS revenue_leakage,
    SUM(mismatch_sessions) AS mismatch_sessions,
    (SELECT COUNT(DISTINCT member_id) FROM {catalog}.gold.fact_sessions WHERE status = 'COMPLETED' AND member_id IS NOT NULL) AS active_members
FROM {catalog}.gold.agg_daily_station;
""")

###E5

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.reconciliation_results (
    run_ts TIMESTAMP,
    check_name STRING,
    source_value DECIMAL(18,3),
    target_value DECIMAL(18,3),
    status STRING
) USING DELTA;
""")

# Execute and append reconciliation checks
spark.sql(f"""
INSERT INTO {catalog}.ops.reconciliation_results
SELECT 
    CURRENT_TIMESTAMP() AS run_ts,
    'SESSION_COUNT' AS check_name,
    CAST((SELECT COUNT(DISTINCT session_id) FROM {catalog}.bronze.sessions) AS DECIMAL(18,3)) AS source_value,
    CAST((SELECT COUNT(*) FROM {catalog}.silver.sessions) + (SELECT COUNT(*) FROM {catalog}.silver.quarantine_sessions) AS DECIMAL(18,3)) AS target_value,
    CASE WHEN (SELECT COUNT(DISTINCT session_id) FROM {catalog}.bronze.sessions) = ((SELECT COUNT(*) FROM {catalog}.silver.sessions) + (SELECT COUNT(*) FROM {catalog}.silver.quarantine_sessions)) THEN 'PASS' ELSE 'FAIL' END AS status

UNION ALL

SELECT 
    CURRENT_TIMESTAMP() AS run_ts,
    'ENERGY_KWH' AS check_name,
    CAST((SELECT SUM(energy_kwh) FROM {catalog}.silver.sessions WHERE status = 'COMPLETED') AS DECIMAL(18,3)) AS source_value,
    CAST((SELECT SUM(energy_kwh) FROM {catalog}.gold.agg_daily_station) AS DECIMAL(18,3)) AS target_value,
    CASE WHEN (SELECT SUM(energy_kwh) FROM {catalog}.silver.sessions WHERE status = 'COMPLETED') = (SELECT SUM(energy_kwh) FROM {catalog}.gold.agg_daily_station) THEN 'PASS' ELSE 'FAIL' END AS status

UNION ALL

SELECT 
    CURRENT_TIMESTAMP() AS run_ts,
    'CURRENT_MEMBERS' AS check_name,
    CAST((SELECT COUNT(DISTINCT member_id) FROM {catalog}.silver.members) AS DECIMAL(18,3)) AS source_value,
    CAST((SELECT COUNT(*) FROM {catalog}.gold.dim_member WHERE is_current = TRUE) AS DECIMAL(18,3)) AS target_value,
    CASE WHEN (SELECT COUNT(DISTINCT member_id) FROM {catalog}.silver.members) = (SELECT COUNT(*) FROM {catalog}.gold.dim_member WHERE is_current = TRUE) THEN 'PASS' ELSE 'FAIL' END AS status;
""")

###H2

In [0]:
dbutils.widgets.text("checkpoint_name", "sessions")
checkpoint_name = dbutils.widgets.get("checkpoint_name")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.ops.checkpoint_counts (
    checkpoint_name STRING,
    table_name STRING,
    row_count BIGINT,
    captured_at TIMESTAMP
) USING DELTA;
""")

tables_to_count = [
    "bronze.sessions",
    "bronze.members",
    "bronze.chargers",
    "silver.sessions",
    "silver.quarantine_sessions",
    "silver.members",
    "silver.chargers",
    "gold.fact_sessions",
    "gold.agg_daily_station"
]

for tbl in tables_to_count:
    count_df = spark.sql(f"SELECT COUNT(*) AS cnt FROM {catalog}.{tbl}")
    row_cnt = count_df.collect()[0]["cnt"]
    
    spark.sql(f"""
        INSERT INTO {catalog}.ops.checkpoint_counts 
        VALUES ('{checkpoint_name}', '{tbl}', {row_cnt}, CURRENT_TIMESTAMP())
    """)